# Benchmark: L2 Correction vs Initial Jacobian Determinant

Investigates how per-pixel correction magnitude (L2 displacement change) correlates
with the initial Jacobian determinant value at each voxel.

**Expected behaviour:** pixels with strongly negative initial Jdet should require
the largest displacements to fix, yielding a strong negative correlation.

## Analyses
1. **Pooled per-pixel scatter** — initial Jdet vs per-pixel `||Δd||` across all cases and modes
2. **Correlation statistics** — Pearson *r* and Spearman *ρ* per mode
3. **Spatial maps** — initial Jdet heatmap alongside per-pixel L2 change (Jac-only mode)
4. **Fraction of L2 in negative-Jdet region** — how much correction mass sits where Jdet < 0
5. **Per-case correlation breakdown** — correlation coefficients per test case

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

from dvfopt import iterative_with_jacobians2, jacobian_det2D
from dvfopt.testcases import (
    SYNTHETIC_CASES,
    RANDOM_DVF_CASES,
    make_deformation,
    make_random_dvf,
)

In [ ]:
THRESHOLD = 0.01

MODES = {
    "Jacobian only":  {"enforce_shoelace": False, "enforce_injectivity": False},
    "Jac + Shoelace": {"enforce_shoelace": True,  "enforce_injectivity": False},
    "Jac + Inject":   {"enforce_shoelace": False, "enforce_injectivity": True},
    "All constraints":{"enforce_shoelace": True,  "enforce_injectivity": True},
}

## Data collection

For each test case and mode we record:
- `jac_init` — `(H, W)` initial Jacobian determinant
- `l2_px`   — `(H, W)` per-pixel displacement change magnitude `||φ_corr − φ_init||`

In [ ]:
def collect_pixel_data(deformation, label):
    """Run all modes on *deformation* and return per-pixel arrays."""
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])   # (2, H, W): [dy, dx]
    jac_init = np.squeeze(jacobian_det2D(phi_init))               # (H, W)

    rec = {"label": label, "jac_init": jac_init, "modes": {}}

    for mode_name, flags in MODES.items():
        phi_corr = iterative_with_jacobians2(
            deformation.copy(), verbose=0, threshold=THRESHOLD, **flags
        )
        delta   = phi_corr - phi_init                               # (2, H, W)
        l2_px   = np.sqrt(delta[0] ** 2 + delta[1] ** 2)           # (H, W)
        total_l2 = float(np.sqrt((delta ** 2).sum()))
        rec["modes"][mode_name] = {"l2_px": l2_px, "total_l2": total_l2}

    return rec

In [ ]:
all_records = []

for key in SYNTHETIC_CASES:
    deformation, _, _ = make_deformation(key)
    label = SYNTHETIC_CASES[key]["title"]
    print(f"Collecting: {label}")
    all_records.append(collect_pixel_data(deformation, label))

for key in RANDOM_DVF_CASES:
    deformation = make_random_dvf(key)
    label = RANDOM_DVF_CASES[key]["title"]
    print(f"Collecting: {label}")
    all_records.append(collect_pixel_data(deformation, label))

---
## 1 — Pooled scatter: initial Jdet vs per-pixel L2 change

Each point is one pixel from one test case.  Density shown via hexbin log-count.
Vertical dashed line at Jdet = 0 (fold boundary).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flat

# Pool jdet once (same for all modes)
jdet_pool = np.concatenate([rec["jac_init"].ravel() for rec in all_records])

for ax, mode_name in zip(axes, MODES):
    l2_pool = np.concatenate([rec["modes"][mode_name]["l2_px"].ravel() for rec in all_records])

    r_val,   _ = stats.pearsonr(jdet_pool, l2_pool)
    rho_val, _ = stats.spearmanr(jdet_pool, l2_pool)

    hb = ax.hexbin(jdet_pool, l2_pool, gridsize=45, cmap="viridis",
                   mincnt=1, bins="log", linewidths=0.2)
    plt.colorbar(hb, ax=ax, label="log₁₀(count)")

    slope, intercept, *_ = stats.linregress(jdet_pool, l2_pool)
    xfit = np.linspace(jdet_pool.min(), jdet_pool.max(), 300)
    ax.plot(xfit, slope * xfit + intercept, "r-", linewidth=2,
            label=f"r = {r_val:.3f}\nρ = {rho_val:.3f}\nR² = {r_val**2:.3f}")

    ax.axvline(0, color="gray", linestyle="--", linewidth=0.9, alpha=0.8)
    ax.set_xlabel("Initial Jacobian Determinant", fontsize=10)
    ax.set_ylabel("Per-pixel ||Δd|| (L2 change)", fontsize=10)
    ax.set_title(mode_name, fontsize=11, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")

fig.suptitle("Initial Jdet vs Per-pixel Correction Magnitude (all cases pooled)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 2 — Correlation statistics table

In [ ]:
jdet_pool = np.concatenate([rec["jac_init"].ravel() for rec in all_records])

header = f"{'Mode':<20s}  {'Pearson r':>10s}  {'R²':>7s}  {'Spearman ρ':>11s}  {'p-value (ρ)':>13s}"
print(header)
print("-" * len(header))

for mode_name in MODES:
    l2_pool  = np.concatenate([rec["modes"][mode_name]["l2_px"].ravel() for rec in all_records])
    r_val,   _ = stats.pearsonr(jdet_pool, l2_pool)
    rho_val, p_s = stats.spearmanr(jdet_pool, l2_pool)
    print(f"  {mode_name:<18s}  {r_val:>10.4f}  {r_val**2:>7.4f}  {rho_val:>11.4f}  {p_s:>13.3e}")

---
## 3 — Spatial maps: initial Jdet alongside per-pixel L2 change

Using **Jacobian-only** mode to isolate the pure effect of initial Jdet on correction magnitude.
Regions with negative initial Jdet (red in left panel) should overlap the
high-correction regions (bright in right panel).

In [ ]:
SPATIAL_MODE = "Jacobian only"

for rec in all_records:
    jac_init = rec["jac_init"]
    l2_px    = rec["modes"][SPATIAL_MODE]["l2_px"]

    # Skip cases where no correction was applied (all l2_px ~ 0)
    if l2_px.max() < 1e-8:
        print(f"Skipping {rec['label']} — no correction applied")
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Left: initial Jdet
    vmin = min(float(jac_init.min()), -0.5)
    vmax = max(float(jac_init.max()), 1.5)
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    im0 = axes[0].imshow(jac_init, cmap="bwr", norm=norm, origin="upper", interpolation="nearest")
    plt.colorbar(im0, ax=axes[0], label="Jdet")
    axes[0].set_title("Initial Jacobian det", fontsize=10)

    # Middle: per-pixel L2 change
    im1 = axes[1].imshow(l2_px, cmap="hot", origin="upper", interpolation="nearest")
    plt.colorbar(im1, ax=axes[1], label="||Δd||")
    axes[1].set_title(f"Per-pixel L2 change\n({SPATIAL_MODE})", fontsize=10)

    # Right: overlay — L2 change contour on top of Jdet
    axes[2].imshow(jac_init, cmap="bwr", norm=norm, origin="upper",
                   interpolation="nearest", alpha=0.7)
    if l2_px.max() > 0:
        axes[2].contour(l2_px, levels=5, colors="black", linewidths=0.6, alpha=0.85)
    axes[2].set_title("Overlay: Jdet (colour) +\nL2-change contours", fontsize=10)

    fig.suptitle(rec["label"], fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 4 — What fraction of total L2 sits in the negative-Jdet region?

Computes `||Δd||` restricted to pixels where `jac_init ≤ 0` as a percentage of the
total `||Δd||` for that case.

In [ ]:
frac_data = {mode: [] for mode in MODES}  # mode -> list of fractions per case
case_labels = []
neg_px_fracs = []  # fraction of pixels that are neg-Jdet (same for all modes)

for rec in all_records:
    mask_neg = rec["jac_init"] <= 0
    n_neg = mask_neg.sum()
    n_total = mask_neg.size
    neg_px_fracs.append(100.0 * n_neg / n_total)
    case_labels.append(rec["label"].replace("Case ", "").replace(" — ", "\n"))

    for mode_name in MODES:
        l2_px = rec["modes"][mode_name]["l2_px"]
        total_sq = (l2_px ** 2).sum()
        neg_sq   = (l2_px[mask_neg] ** 2).sum()
        frac = 100.0 * float(np.sqrt(neg_sq)) / float(np.sqrt(total_sq)) if total_sq > 0 else 0.0
        frac_data[mode_name].append(frac)

# Bar chart
x = np.arange(len(case_labels))
width = 0.18
mode_list = list(MODES.keys())

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(14, 9),
                                      gridspec_kw={"height_ratios": [3, 1]})

for i, mode_name in enumerate(mode_list):
    ax_top.bar(x + i * width, frac_data[mode_name], width, label=mode_name)

ax_top.set_ylabel("% of total L2 in neg-Jdet pixels", fontsize=11)
ax_top.set_title("Fraction of Correction Mass in Negative-Jdet Region",
                 fontsize=12, fontweight="bold")
ax_top.set_xticks(x + width * 1.5)
ax_top.set_xticklabels(case_labels, rotation=30, ha="right", fontsize=8)
ax_top.axhline(100, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax_top.set_ylim(0, 115)
ax_top.legend(fontsize=9)
ax_top.grid(True, axis="y", alpha=0.3)

# Bottom: fraction of pixels that are negative-Jdet
ax_bot.bar(x + width * 1.5, neg_px_fracs, width * 4, color="#e74c3c", alpha=0.75)
ax_bot.set_ylabel("% pixels\nneg-Jdet", fontsize=9)
ax_bot.set_xticks(x + width * 1.5)
ax_bot.set_xticklabels(case_labels, rotation=30, ha="right", fontsize=8)
ax_bot.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Numeric summary
print(f"\n{'Case':<35s}  {'neg-px%':>7s}", end="")
for m in mode_list:
    print(f"  {m[:14]:>14s}", end="")
print()
print("-" * (35 + 9 + 16 * len(mode_list)))
for i, rec in enumerate(all_records):
    print(f"  {rec['label']:<33s}  {neg_px_fracs[i]:>6.1f}%", end="")
    for m in mode_list:
        print(f"  {frac_data[m][i]:>13.1f}%", end="")
    print()

---
## 5 — Per-case Pearson / Spearman correlation breakdown

In [ ]:
corr_rows = []  # (case, mode, r, rho)

for rec in all_records:
    jdet_flat = rec["jac_init"].ravel()
    for mode_name in MODES:
        l2_flat = rec["modes"][mode_name]["l2_px"].ravel()
        if l2_flat.std() < 1e-10:
            corr_rows.append((rec["label"], mode_name, float("nan"), float("nan")))
            continue
        r_val,   _ = stats.pearsonr(jdet_flat, l2_flat)
        rho_val, _ = stats.spearmanr(jdet_flat, l2_flat)
        corr_rows.append((rec["label"], mode_name, r_val, rho_val))

# Print table
print(f"{'Case':<35s}  {'Mode':<18s}  {'Pearson r':>10s}  {'Spearman ρ':>11s}")
print("-" * 80)
prev_case = None
for case, mode, r, rho in corr_rows:
    if case != prev_case and prev_case is not None:
        print()
    prev_case = case
    r_str   = f"{r:>10.4f}"   if not np.isnan(r)   else f"{'N/A':>10s}"
    rho_str = f"{rho:>11.4f}" if not np.isnan(rho) else f"{'N/A':>11s}"
    print(f"  {case:<33s}  {mode:<18s}  {r_str}  {rho_str}")

---
## 6 — Correlation strength vs severity of folding

Each point is one test case.  x = minimum initial Jdet (most folded pixel),
y = total L2 correction applied.  Tests whether more severely folded fields
require disproportionately larger corrections.

In [ ]:
fig, axes = plt.subplots(1, len(MODES), figsize=(5 * len(MODES), 4), sharey=False)

for ax, mode_name in zip(axes, MODES):
    min_jdets  = [float(rec["jac_init"].min()) for rec in all_records]
    total_l2s  = [rec["modes"][mode_name]["total_l2"] for rec in all_records]
    labels_s   = [rec["label"] for rec in all_records]

    ax.scatter(min_jdets, total_l2s, s=60, zorder=3)

    for xi, yi, lbl in zip(min_jdets, total_l2s, labels_s):
        ax.annotate(lbl.split("\u2014")[-1].strip()[:14],
                    (xi, yi), fontsize=6, ha="left",
                    xytext=(3, 2), textcoords="offset points")

    if len(set(total_l2s)) > 1:
        slope, intercept, r, *_ = stats.linregress(min_jdets, total_l2s)
        xfit = np.linspace(min(min_jdets), max(min_jdets), 200)
        ax.plot(xfit, slope * xfit + intercept, "r--", linewidth=1.5,
                label=f"r = {r:.3f}")
        ax.legend(fontsize=9)

    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.set_xlabel("Min initial Jdet (per case)", fontsize=9)
    ax.set_ylabel("Total L2 correction", fontsize=9)
    ax.set_title(mode_name, fontsize=10, fontweight="bold")
    ax.grid(True, alpha=0.3)

fig.suptitle("Case-level: Folding Severity vs Total L2 Correction",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()